In [3]:
import sys
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
from numba import njit, prange
from matplotlib.animation import FuncAnimation,FFMpegWriter, PillowWriter
from IPython.display import HTML
from scipy.interpolate import RegularGridInterpolator
import time
import pandas as pd
from copy import deepcopy
import seaborn as sns
from numba import set_num_threads
from itertools import product
from tqdm import tqdm
from collections import defaultdict
import copy
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "font.size": 20,
    "axes.labelsize": 20,
    "axes.titlesize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 20,
    # No uses font.weight aquí: no afecta cuando usas LaTeX
})


set_num_threads(14)  # Reemplaza N con el número de núcleos que quieres usar

# Discretización y proceso de simulación

Consideramos el sistema adimensional de ecuaciones de Navier–Stokes incompresibles con energía:

$$
\nabla \cdot \mathbf{u} = 0,
$$

$$
\frac{\partial \mathbf{u}}{\partial t} + (\mathbf{u}\cdot\nabla)\mathbf{u} = 
-\, \nabla p + \frac{1}{Re} \nabla^2 \mathbf{u},
$$

$$
\frac{\partial T}{\partial t} + (\mathbf{u}\cdot\nabla)T = 
\frac{1}{Re\,Pr} \nabla^2 T + \frac{Ec}{Re} \, \Phi,
$$

donde $\mathbf{u} = (u,v)$ es el campo de velocidades, $p$ la presión,
$T$ la temperatura, y $\Phi$ el término de disipación viscosa:

$$
\Phi = 2\left(\left(\frac{\partial u}{\partial x}\right)^2 + 
\left(\frac{\partial v}{\partial y}\right)^2\right) + 
4\left(\tfrac{1}{2}\Big(\tfrac{\partial u}{\partial y} + \tfrac{\partial v}{\partial x}\Big)\right)^2.
$$

---

## Discretización espacial

Malla cartesiana uniforme $(x_i,y_j)$ con pasos

$$
\Delta x = \frac{L_x}{n_x-1}, \qquad \Delta y = \frac{L_y}{n_y-1}.
$$

Diferencias centradas de segundo orden:

$$
\left.\frac{\partial \phi}{\partial x}\right|_{i,j} \approx \frac{\phi_{i,j+1} - \phi_{i,j-1}}{2\Delta x},
$$

$$
\left.\frac{\partial^2 \phi}{\partial x^2}\right|_{i,j} \approx \frac{\phi_{i,j+1} - 2\phi_{i,j} + \phi_{i,j-1}}{\Delta x^2},
$$

y de forma análoga en $y$.

---

## Discretización temporal

Euler explícito:

$$
\phi^{n+1}_{i,j} = \phi^n_{i,j} + \Delta t \, \mathcal{RHS}(\phi^n).
$$

---

## Esquema de proyección

1. **Predicción sin presión:**

$$
u^*_{i,j} = u^n_{i,j} + \Delta t \left(-\big[u\partial_x u + v\partial_y u\big]_{i,j} + \tfrac{1}{Re}\nabla^2 u_{i,j}\right),
$$

$$
v^*_{i,j} = v^n_{i,j} + \Delta t \left(-\big[u\partial_x v + v\partial_y v\big]_{i,j} + \tfrac{1}{Re}\nabla^2 v_{i,j}\right).
$$

2. **Poisson para la presión:**

$$
\nabla^2 p^{n+1}_{i,j} = \frac{1}{\Delta t}\left(\frac{u^*_{i,j+1} - u^*_{i,j-1}}{2\Delta x} + \frac{v^*_{i+1,j} - v^*_{i-1,j}}{2\Delta y}\right).
$$

3. **Corrección de velocidades:**

$$
u^{n+1}_{i,j} = u^*_{i,j} - \Delta t \, \frac{p^{n+1}_{i,j+1} - p^{n+1}_{i,j-1}}{2\Delta x},
$$

$$
v^{n+1}_{i,j} = v^*_{i,j} - \Delta t \, \frac{p^{n+1}_{i+1,j} - p^{n+1}_{i-1,j}}{2\Delta y}.
$$

---

## Ecuación de energía

$$
T^{n+1}_{i,j} = T^n_{i,j} + \Delta t\left(
- \big[u \partial_x T + v \partial_y T\big]_{i,j} 
+ \frac{1}{Re\,Pr} \nabla^2 T_{i,j} 
+ \frac{Ec}{Re} \, \Phi_{i,j}
\right).
$$

---

## Proceso de simulación iterativa

1. Aplicar condiciones de frontera a $u^n, v^n, T^n$.
2. Calcular los predictores $u^*, v^*$.
3. Resolver Poisson para $p^{n+1}$.
4. Corregir velocidades $\rightarrow u^{n+1}, v^{n+1}$.
5. Actualizar temperatura $T^{n+1}$.
6. Guardar resultados si corresponde.


# Funcion general de simulacion

In [ ]:
def fluid_params(fluid="air", U=0.1, L=0.1, dT=10.0):
    """
    Retorna Re, Pr, Ec para un fluido dado en condiciones normales.
    fluid: "air", "water", "oil"
    U, L: escalas de velocidad y longitud
    dT: escala de temperatura
    dp: escala de presión (si no se conoce, usa ρ U^2 -> Eu≈1)
    """
    props = {
        "air":   {"rho": 1.2,  "mu": 1.8e-5, "k": 0.026, "cp": 1005.0},
        "water": {"rho": 998., "mu": 1.0e-3, "k": 0.6,   "cp": 4182.0},
        "oil":   {"rho": 870., "mu": 0.29,   "k": 0.145, "cp": 2000.0}
    }
    f = props[fluid]
    rho, mu, k, cp = f["rho"], f["mu"], f["k"], f["cp"]

    Re = rho*U*L/mu
    Pr = mu*cp/k
    Ec = U**2/(cp*dT)

    return Re, Pr, Ec



In [ ]:
@njit(parallel=True, fastmath=True)
def actualizar_campos(u_old, v_old, p_old, T_old, 
                      u_star, v_star, p_star, T_star,
                      Re, Pr, Ec, Eu, dt_star, dx_star, dy_star, nx, ny,
                      n_poisson=60):
    dx2 = dx_star * dx_star
    dy2 = dy_star * dy_star

    # -------------------------
    # 1) Predictor sin presión
    # -------------------------
    for i in prange(1, ny - 1):
        for j in range(1, nx - 1):
            # u
            conv_u_x = u_old[i, j] * (u_old[i, j + 1] - u_old[i, j - 1]) / (2.0 * dx_star)
            conv_u_y = v_old[i, j] * (u_old[i + 1, j] - u_old[i - 1, j]) / (2.0 * dy_star)
            diff_u_x = (u_old[i, j + 1] - 2.0 * u_old[i, j] + u_old[i, j - 1]) / dx2
            diff_u_y = (u_old[i + 1, j] - 2.0 * u_old[i, j] + u_old[i - 1, j]) / dy2
            u_star[i, j] = u_old[i, j] + dt_star * (-conv_u_x - conv_u_y + (1.0 / Re) * (diff_u_x + diff_u_y))

            # v
            conv_v_x = u_old[i, j] * (v_old[i, j + 1] - v_old[i, j - 1]) / (2.0 * dx_star)
            conv_v_y = v_old[i, j] * (v_old[i + 1, j] - v_old[i - 1, j]) / (2.0 * dy_star)
            diff_v_x = (v_old[i, j + 1] - 2.0 * v_old[i, j] + v_old[i, j - 1]) / dx2
            diff_v_y = (v_old[i + 1, j] - 2.0 * v_old[i, j] + v_old[i - 1, j]) / dy2
            v_star[i, j] = v_old[i, j] + dt_star * (-conv_v_x - conv_v_y + (1.0 / Re) * (diff_v_x + diff_v_y))

    # BCs provisionales en u,v (antes de Poisson), coherentes con tu caso
    v_star[0, :]  = 0.0
    v_star[-1, :] = 0.0
    u_star[0, :]  = -0.5
    u_star[-1, :] =  1.0
    # Entrada/salida Neumann
    u_star[:, 0]  = u_star[:, 1]
    u_star[:, -1] = u_star[:, -2]
    v_star[:, 0]  = v_star[:, 1]
    v_star[:, -1] = v_star[:, -2]

    # -------------------------
    # 2) Poisson para presión
    #     ∇²p = (1/Δt) div(u*)
    # -------------------------
    for _ in range(n_poisson):
        p_tmp = p_star.copy()
        for i in prange(1, ny - 1):
            for j in range(1, nx - 1):
                div = (u_star[i, j + 1] - u_star[i, j - 1]) / (2.0 * dx_star) + \
                      (v_star[i + 1, j] - v_star[i - 1, j]) / (2.0 * dy_star)
                rhs = (1.0 / dt_star) * div
                p_star[i, j] = ((p_tmp[i + 1, j] + p_tmp[i - 1, j]) * dy2 +
                                (p_tmp[i, j + 1] + p_tmp[i, j - 1]) * dx2 -
                                rhs * dx2 * dy2) / (2.0 * (dx2 + dy2))

        # BCs de presión: Neumann en bordes
        for i in prange(ny):
            p_star[i, 0]  = p_star[i, 1]
            p_star[i, -1] = p_star[i, -2]
        for j in prange(nx):
            p_star[0, j]  = p_star[1, j]
            p_star[-1, j] = p_star[-2, j]

        # Fijar referencia (evita singularidad)
        p_star[0, 0] = 0.0

    # -------------------------
    # 3) Proyección (corrección)
    # -------------------------
    for i in prange(1, ny - 1):
        for j in range(1, nx - 1):
            u_star[i, j] -= dt_star * Eu * (p_star[i, j + 1] - p_star[i, j - 1]) / (2.0 * dx_star)
            v_star[i, j] -= dt_star * Eu * (p_star[i + 1, j] - p_star[i - 1, j]) / (2.0 * dy_star)

    # BCs finales en velocidades (igual que arriba)
    v_star[0, :]  = 0.0
    v_star[-1, :] = 0.0
    u_star[0, :]  = -0.5
    u_star[-1, :] =  1.0
    u_star[:, 0]  = u_star[:, 1]
    u_star[:, -1] = u_star[:, -2]
    v_star[:, 0]  = v_star[:, 1]
    v_star[:, -1] = v_star[:, -2]

    # -------------------------
    # 4) Energía (temperatura)
    # -------------------------
    for i in prange(1, ny - 1):
        for j in range(1, nx - 1):
            conv_T_x = u_star[i, j] * (T_old[i, j + 1] - T_old[i, j - 1]) / (2.0 * dx_star)
            conv_T_y = v_star[i, j] * (T_old[i + 1, j] - T_old[i - 1, j]) / (2.0 * dy_star)
            diff_T_x = (T_old[i, j + 1] - 2.0 * T_old[i, j] + T_old[i, j - 1]) / dx2
            diff_T_y = (T_old[i + 1, j] - 2.0 * T_old[i, j] + T_old[i - 1, j]) / dy2
            Sxx = (u_star[i, j + 1] - u_star[i, j - 1]) / (2.0 * dx_star)
            Syy = (v_star[i + 1, j] - v_star[i - 1, j]) / (2.0 * dy_star)
            Sxy = 0.5 * ((u_star[i + 1, j] - u_star[i - 1, j]) / (2.0 * dy_star) +
                         (v_star[i, j + 1] - v_star[i, j - 1]) / (2.0 * dx_star))
            viscous_heating = (Ec / Re) * (2.0 * (Sxx*Sxx + Syy*Syy) + 4.0 * Sxy*Sxy)

            T_star[i, j] = T_old[i, j] + dt_star * (
                -conv_T_x - conv_T_y + (1.0 / (Re * Pr)) * (diff_T_x + diff_T_y) + viscous_heating
            )

    # (opcional) Esfuerzo cortante tau_yx ≈ ∂u/∂y
    tau = np.zeros((ny, nx))
    for i in prange(1, ny - 1):
        for j in range(nx):
            tau[i, j] = (u_star[i + 1, j] - u_star[i - 1, j]) / (2.0 * dy_star)

    return u_star, v_star, p_star, T_star, tau


In [ ]:
def run_simulacion_general(nx=25, ny=25,
                           observaciones_dict=None,
                           metodo_asimilacion=None,
                           variables_asimilar=['T'],
                           metodo_kwargs={},
                           N_FRAMES=300):

    Re, Pr, Ec, Eu = 20.0, 10.0, 0.1, 1.0
    Lx_star = Ly_star = 1.0
    dx_star = Lx_star / (nx - 1)
    dy_star = Ly_star / (ny - 1)

    # CFL convectivo sencillo (seguro)
    U_ref = 1.0
    dt_star = 0.8 * min(dx_star, dy_star) / U_ref
    t_final = 2.0
    nt = int(np.ceil(t_final / dt_star))
    dt_star = t_final / nt  # reajuste exacto

    save_interval = max(1, nt // N_FRAMES)
    save_times = set(range(0, nt, save_interval))
    save_times.add(nt - 1)

    x_star = np.linspace(0.0, Lx_star, nx)
    y_star = np.linspace(0.0, Ly_star, ny)
    X_star, Y_star = np.meshgrid(x_star, y_star)

    # Condiciones iniciales
    u0, up = 1.0, -0.5
    T0_star, T1_star = 0.0, 0.0
    u_star = np.ones((ny, nx)) * u0
    v_star = np.zeros((ny, nx))
    p_star = np.zeros((ny, nx))
    T_star = np.ones((ny, nx)) * T1_star
    u_star[0, :]  = up
    u_star[-1, :] = u0
    T_star[0, :]  = T0_star
    T_star[-1, :] = T1_star

    u_hist, v_hist, p_hist, T_hist, tau_hist = [], [], [], [], []
    total_assim_time = 0.0

    for n in range(nt):
        u_old, v_old, p_old, T_old = u_star.copy(), v_star.copy(), p_star.copy(), T_star.copy()

        u_star, v_star, p_star, T_star, tau = actualizar_campos(
            u_old, v_old, p_old, T_old,
            u_star, v_star, p_star, T_star,
            Re, Pr, Ec, Eu, dt_star, dx_star, dy_star, nx, ny
        )

        # Asimilación opcional
        if observaciones_dict and (n in observaciones_dict) and (metodo_asimilacion is not None):
            t0 = time.time()
            obs_t = observaciones_dict[n]
            for var in variables_asimilar:
                if var in obs_t:
                    campo = {'T': T_star, 'p': p_star, 'tau': tau}[var]
                    campo_asim = metodo_asimilacion(
                        campo_modelo=campo,
                        observaciones=obs_t,
                        variable=var,
                        nx=nx, ny=ny,
                        **metodo_kwargs
                    )
                    if var == 'T':   T_star = campo_asim
                    elif var == 'p': p_star = campo_asim
                    elif var == 'tau': tau = campo_asim
            total_assim_time += time.time() - t0

        # Guardado en memoria
        if n in save_times:
            u_hist.append(u_star.copy())
            v_hist.append(v_star.copy())
            p_hist.append(p_star.copy())
            T_hist.append(T_star.copy())
            tau_hist.append(tau.copy())

    print("\n✅ Simulación finalizada.")

    return {
        "u_history": u_hist,
        "v_history": v_hist,
        "p_history": p_hist,
        "T_history": T_hist,
        "tau_history": tau_hist,
        "params": {
            "nx": nx, "ny": ny, "dt_star": dt_star, "nt": nt,
            "metodo_asimilacion": metodo_asimilacion.__name__ if metodo_asimilacion else None,
            "variables_asimilar": variables_asimilar
        },
        "X_star": X_star,
        "Y_star": Y_star,
        "asimilacion_time": total_assim_time
    }


In [ ]:
EJECUTAR_SIMULACIONES = True  # 🔁 Cambia a True si quieres que esta celda se ejecute

if EJECUTAR_SIMULACIONES:

    # --- Asegura que existe la carpeta de salida ---
    os.makedirs("sim_modelo_simple", exist_ok=True)

    # --- Lista de resoluciones (lado x = lado y) ---
    resoluciones = [25]

    # --- Simulación para cada resolución ---
    for res in tqdm(resoluciones, desc="Generando simulaciones"):
        print(f"\n🔧 Resolución: {res}x{res}")

        sim_data = run_simulacion_general(nx=res, ny=res,
                           observaciones_dict=None,
                           metodo_asimilacion=None,
                           variables_asimilar=['T'],
                           metodo_kwargs={},
                           N_FRAMES=300)

        ruta_salida = f"sim_modelo_simple/{res}.pkl"
        with open(ruta_salida, "wb") as f:
            pickle.dump(sim_data, f)

        print(f"✅ Guardado: {ruta_salida}")


Generando simulaciones:   0%|          | 0/1 [00:00<?, ?it/s]


🔧 Resolución: 50x50


Generando simulaciones: 100%|██████████| 1/1 [00:04<00:00,  4.53s/it]


✅ Simulación finalizada.
✅ Guardado: sim_modelo_simple/50.pkl


In [76]:
def plot_4_frames_velocidad(path_simulacion, frames=[0, 20, 150, -1], cmap="jet"):
    """
    Representa 4 frames de la magnitud de la velocidad |u| en un grid 2x2:
    inicio, dos intermedios y final.
    Muestra t en los títulos y dibuja líneas de malla acorde a la resolución.
    """
    import pickle
    import numpy as np
    import matplotlib.pyplot as plt

    # --- Cargar simulación ---
    with open(path_simulacion, "rb") as f:
        sim = pickle.load(f)

    X_star = sim["X_star"]
    Y_star = sim["Y_star"]
    u_hist = np.array(sim["u_history"])
    v_hist = np.array(sim["v_history"])
    vel_mag = np.sqrt(u_hist**2 + v_hist**2)

    # Reconstruir el tiempo t (normalizado de 0 a 1)
    n_frames = len(vel_mag)
    time_star = np.linspace(0, 1, n_frames)

    x_vals = X_star[0, :]
    y_vals = Y_star[:, 0]

    # Extraer los frames deseados y preparar los títulos
    selected_frames = [vel_mag[i] if i != -1 else vel_mag[-1] for i in frames]
    frame_indices = [i if i != -1 else n_frames - 1 for i in frames]
    titles = [f"$t = {time_star[i]:.2f}$" for i in frame_indices]

    # --- Crear figura en grid 2x2 ---
    fig, axs = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
    axs = axs.flatten()

    for ax, frame_data, title in zip(axs, selected_frames, titles):
        cf = ax.contourf(X_star, Y_star, frame_data, levels=30, cmap=cmap)

        # Dibujar líneas de malla
        for x in x_vals:
            ax.axvline(x, color='black', linestyle='--', linewidth=0.5, alpha=0.5)
        for y in y_vals:
            ax.axhline(y, color='black', linestyle='--', linewidth=0.5, alpha=0.5)

        # Estética
        ax.set_title(title, fontsize=26)
        ax.set_xlabel(r"$x$", fontsize=26)
        ax.set_ylabel(r"$y$", fontsize=26)
        ax.tick_params(axis='both', which='major', labelsize=20)

        # Reducir densidad de ticks
        nx = len(x_vals)
        ny = len(y_vals)
        xticks_idx = np.linspace(0, nx - 1, 5, dtype=int)
        yticks_idx = np.linspace(0, ny - 1, 5, dtype=int)
        ax.set_xticks(x_vals[xticks_idx])
        ax.set_yticks(y_vals[yticks_idx])

    # Colorbar común
    cbar = fig.colorbar(cf, ax=axs, shrink=0.95)
    cbar.set_label(r"$|u|$", fontsize=26)
    cbar.ax.tick_params(labelsize=20)
    cbar.set_ticks(np.linspace(0, 1, 5))


    plt.show()

In [92]:
archivo_sim = "sim_modelo_simple/50.pkl"
plot_4_frames_velocidad(archivo_sim, frames=[np.linspace(0, len(archivo_sim['u_history']), 4, dtype=int)])

TypeError: string indices must be integers, not 'str'

# Normalizacion de los datos

In [ ]:
def normalizar_simulaciones(carpeta_entrada="sim_dif_res", carpeta_salida="sim_normal"):
    os.makedirs(carpeta_salida, exist_ok=True)

    # --- 1. Buscar archivos .pkl ---
    archivos = [f for f in os.listdir(carpeta_entrada) if f.endswith(".pkl")]
    if not archivos:
        print("❌ No se encontraron archivos .pkl en la carpeta de entrada.")
        return

    print(f"🔍 Archivos encontrados: {archivos}")

    # --- 2. Extraer valores globales ---
    all_vals = {"T": [], "p": [], "tau": []}
    sims = {}

    for archivo in archivos:
        with open(os.path.join(carpeta_entrada, archivo), "rb") as f:
            sim = pickle.load(f)
            sims[archivo] = sim

            for var in ["T", "p", "tau"]:
                hist = sim.get(f"{var}_history", [])
                for frame in hist:
                    all_vals[var].extend(frame.ravel())

    # --- 3. Calcular el valor absoluto máximo global ---
    norm_info = {}
    for var in ["T", "p", "tau"]:
        arr = np.array(all_vals[var])
        max_abs = np.max(np.abs(arr))
        if max_abs == 0:
            print(f"⚠️ {var}: Todos los valores son cero. Se omite normalización.")
            norm_info[var] = None
        else:
            norm_info[var] = max_abs
            print(f"🔧 Normalización {var}: max |valor| = {max_abs:.6f}")

    # --- 4. Aplicar normalización y guardar ---
    for archivo, sim in sims.items():
        sim_norm = copy.deepcopy(sim)
        for var in ["T", "p", "tau"]:
            if norm_info[var] is not None:
                max_abs = norm_info[var]
                sim_norm[f"{var}_history"] = [
                    f / max_abs for f in sim[f"{var}_history"]
                ]
        salida_path = os.path.join(carpeta_salida, archivo)
        with open(salida_path, "wb") as f:
            pickle.dump(sim_norm, f)
        print(f"✅ Guardado normalizado: {salida_path}")

    print("🎯 Todas las simulaciones fueron normalizadas y guardadas (preservando signo).")


In [ ]:
EJECUTAR_SIMULACIONES = False  # 🔁 Cambia a True si quieres que esta celda se ejecute

if EJECUTAR_SIMULACIONES:
    # Ejecutar la normalización global sobre los datos en sim_dif_res
    normalizar_simulaciones(
        carpeta_entrada="sim_dif_res",
        carpeta_salida="sim_normal"
    )

## 🧠 Asimilación de Datos

Se ejecutan distintas configuraciones de asimilación usando `nudging`, `Kalman Filter`, `EnKF`, y `4DVar`.

Los resultados se analizan calculando el `RMSE` con respecto a la referencia interpolada.


# Binning datos de 800x800 a 25x25

In [ ]:
def downsample_por_bloques(frames, factor):
    frames = np.array(frames)
    n_frames, H, W = frames.shape
    h_new, w_new = H // factor, W // factor
    frames = frames[:, :h_new*factor, :w_new*factor]  # Recortar si no divisible
    frames_reducidos = frames.reshape(n_frames, h_new, factor, w_new, factor)
    return frames_reducidos.mean(axis=(2, 4))

def reducir_por_bloques_simulacion(
    archivo_origen="sim_normal/800.pkl",
    carpeta_destino="sim_normal",
    resolucion_objetivo=25
):
    factor = 800 // resolucion_objetivo
    if 800 % resolucion_objetivo != 0:
        raise ValueError("⚠️ La resolución 800 no es divisible por la resolución objetivo.")

    with open(archivo_origen, "rb") as f:
        sim = pickle.load(f)

    # Reducción por bloques para todas las variables
    sim_reducido = {
        "T_history": downsample_por_bloques(sim["T_history"], factor),
        "p_history": downsample_por_bloques(sim["p_history"], factor),
        "tau_history": downsample_por_bloques(sim["tau_history"], factor),
        "u_history": downsample_por_bloques(sim["u_history"], factor),
        "v_history": downsample_por_bloques(sim["v_history"], factor),
    }

    # Crear nuevas mallas X, Y reducidas
    x_physical = np.linspace(0, 1, resolucion_objetivo)
    y_physical = np.linspace(0, 1, resolucion_objetivo)
    X_star, Y_star = np.meshgrid(x_physical, y_physical)

    sim_reducido["X_star"] = X_star
    sim_reducido["Y_star"] = Y_star
    sim_reducido["params"] = {
        "nx": resolucion_objetivo,
        "ny": resolucion_objetivo,
        "fuente": "downsample_por_bloques_desde_800x800"
    }
    sim_reducido["descripcion"] = f"Reducción por bloques desde 800x800 a {resolucion_objetivo}x{resolucion_objetivo}"

    # Guardar
    os.makedirs(carpeta_destino, exist_ok=True)
    nombre_archivo = f"{resolucion_objetivo}_bloques.pkl"
    with open(os.path.join(carpeta_destino, nombre_archivo), "wb") as f:
        pickle.dump(sim_reducido, f)

    print(f"✅ Guardado archivo reducido por bloques en: {os.path.join(carpeta_destino, nombre_archivo)}")


In [ ]:
EJECUTAR_SIMULACIONES = False  # 🔁 Cambia a True si quieres que esta celda se ejecute

if EJECUTAR_SIMULACIONES:
    reducir_por_bloques_simulacion(
        archivo_origen="sim_normal/800.pkl",
        carpeta_destino="sim_normal",
        resolucion_objetivo=25
    )


✅ Guardado archivo reducido por bloques en: sim_normal/25_bloques.pkl


# Observaciones

In [ ]:
def extraer_observaciones(
    path_pkl,
    N_sens=10,
    T_sens=5,
    ruido_std=0.0,
    seed=None
):
    """
    Extrae observaciones desde un archivo .pkl de simulación (normalizado o no).

    Parámetros:
    - path_pkl: ruta al archivo .pkl con los datos simulados/interpolados
    - N_sens: número de sensores a lo largo de x
    - T_sens: número de pasos temporales donde tomar observaciones
    - ruido_std: desviación estándar del ruido (gaussiano)
    - seed: semilla para reproducibilidad

    Retorna:
    - Lista de diccionarios con observaciones por instante
    """

    if seed is not None:
        np.random.seed(seed)

    with open(path_pkl, "rb") as f:
        datos = pickle.load(f)

    T_hist = datos["T_history"]
    p_hist = datos["p_history"]
    tau_hist = datos["tau_history"]
    Y_star = datos["Y_star"]

    nt = len(T_hist)
    nx = T_hist[0].shape[1]
    x_physical = np.linspace(0, 1, N_sens)
    x_idx = (x_physical * (nx - 1)).astype(int)
    frame_indices = np.linspace(0, nt - 1, T_sens, dtype=int)

    # Elegir fila más cercana a y = 0.001
    y_fila = Y_star[:, 0]
    y_idx = int(np.argmin(np.abs(y_fila - 0.001)))

    observaciones = []
    for t in frame_indices:
        T_line = T_hist[t][y_idx, :]
        p_line = p_hist[t][y_idx, :]
        tau_line = tau_hist[t][y_idx, :]

        obs_t = {
            "t": t,
            "x_phys": x_physical,
            "T": T_line[x_idx].copy(),
            "p": p_line[x_idx].copy(),
            "tau": tau_line[x_idx].copy()
        }

        if ruido_std > 0.0:
            obs_t["T"] += np.random.normal(0, ruido_std, size=N_sens)
            obs_t["p"] += np.random.normal(0, ruido_std, size=N_sens)
            obs_t["tau"] += np.random.normal(0, ruido_std, size=N_sens)

        observaciones.append(obs_t)

    return observaciones


# Nudging

In [ ]:
def aplicar_nudging(campo_modelo, observaciones, variable, nx, ny, alpha=0.1):
    """
    Aplica el método de Nudging para corregir un campo usando observaciones puntuales en y=0.

    Parámetros:
    -----------
    campo_modelo : ndarray
        Campo actual del modelo (T, p o tau), de tamaño (ny, nx).

    observaciones : dict
        Diccionario con las observaciones en este timestep.
        Debe contener claves como 'T', 'p', 'tau' y 'x_phys'.

    variable : str
        Nombre de la variable a asimilar: 'T', 'p' o 'tau'.

    nx, ny : int
        Tamaño espacial del dominio.

    alpha : float
        Coeficiente de nudging (cuánto se ajusta hacia la observación).

    Retorna:
    --------
    ndarray con el campo corregido.
    """
    campo_corr = campo_modelo.copy()

    if variable not in observaciones or 'x_phys' not in observaciones:
        return campo_corr

    x_phys = np.array(observaciones['x_phys'])
    x_idx = (x_phys * (nx - 1)).astype(int)
    y_idx = np.zeros_like(x_idx)  # Asumimos observaciones en y=0

    for xi, yi, obs_val in zip(x_idx, y_idx, observaciones[variable]):
        campo_corr[yi, xi] = (1 - alpha) * campo_corr[yi, xi] + alpha * obs_val

    return campo_corr


# Kalman filter

In [ ]:
def aplicar_kalman_filter(campo_modelo, observaciones, variable, nx, ny,
                          sigma_obs=0.01, sigma_modelo=1e-4):
    """
    Aplica el filtro de Kalman clásico (KF) para una variable en y=0.

    Parámetros:
    -----------
    campo_modelo : np.ndarray
        Campo actual del modelo (e.g., T_star, p_star, tau).

    observaciones : dict
        Diccionario con observaciones. Debe incluir:
        - 'x_phys': posiciones físicas de los sensores
        - variable: valores observados para esa variable

    variable : str
        Nombre de la variable a asimilar ('T', 'p' o 'tau').

    nx, ny : int
        Dimensiones del dominio.

    sigma_obs : float
        Desviación estándar del ruido de observación.

    sigma_modelo : float
        Incertidumbre del modelo (ruido del sistema).

    Retorna:
    --------
    campo_actualizado : np.ndarray
        Campo corregido en base a las observaciones.
    """
    campo_actualizado = campo_modelo.copy()
    x_phys = observaciones["x_phys"]
    obs_vals = observaciones[variable]

    x_idx = (x_phys * (nx - 1)).astype(int)
    y_idx = np.zeros_like(x_idx)  # sensores en y = 0

    # Extraer predicciones del modelo
    x_fondo = campo_modelo[y_idx, x_idx]

    # Construir matrices
    H = np.eye(len(x_idx))  # observación directa
    R = np.eye(len(x_idx)) * sigma_obs**2
    P = np.eye(len(x_idx)) * sigma_modelo**2

    # Ganancia de Kalman
    K = P @ np.linalg.inv(P + R)

    # Corrección
    x_actualizado = x_fondo + K @ (obs_vals - H @ x_fondo)

    # Insertar corrección en el campo
    for xi, yi, val in zip(x_idx, y_idx, x_actualizado):
        campo_actualizado[yi, xi] = val

    return campo_actualizado


# Ensemble Kalman filter

Necesita su propia simulacion porque se añaden varios 'hilos' de simulacion. No solo se usa un valor inicial y se simula su evolucion (teniendo en cuenta la asimilacion), sino que genera N valores iniciales y simula su evolucion simultaneamente, formando el ensemble.

In [ ]:
def run_enkf(nx=25, ny=25, N_ens=10, presion_adversa=0.3,
             observaciones_dict=None,
             variables_asimilar=['T'],
             sigma_obs=0.01, sigma_modelo=1e-4,
             N_FRAMES=300, seed=None):
    """
    Simulación con Ensemble Kalman Filter (EnKF).

    Parámetros:
    -----------
    N_ens : int
        Número de miembros del ensamble.
    sigma_obs : float
        Desviación estándar del ruido de observación.
    """
    if seed is not None:
        np.random.seed(seed)

    # Parámetros del modelo
    Re, Pr, Ec, Eu = 20.0, 10.0, 0.1, 1.0
    Lx_star = Ly_star = 1.0
    dx_star = Lx_star / (nx - 1)
    dy_star = Ly_star / (ny - 1)
    dt_cfl = 0.008 * dx_star / 1.0
    nt = int(np.ceil(2.0 / dt_cfl))
    dt_star = 1.0 / nt

    save_interval = max(1, nt // N_FRAMES)
    save_times = set(range(0, nt, save_interval))
    save_times.add(nt - 1)

    x_star = np.linspace(0, Lx_star, nx)
    y_star = np.linspace(0, Ly_star, ny)
    X_star, Y_star = np.meshgrid(x_star, y_star)

    # Estado inicial
    def crear_estado():
        u = np.ones((ny, nx)) * 1.0
        v = np.zeros((ny, nx))
        p = np.zeros((ny, nx))
        T = np.zeros((ny, nx))
        tau = np.zeros((ny, nx))
        u[0, :] = -0.5
        u[-1, :] = 1.0
        return {'u': u, 'v': v, 'p': p, 'T': T, 'tau': tau}

    ensamble = [crear_estado() for _ in range(N_ens)]

    u_hist, v_hist, p_hist, T_hist, tau_hist = [], [], [], [], []
    total_assim_time = 0.0

    for n in range(nt):
        for e in range(N_ens):
            m = ensamble[e]
            u, v, p, T, tau = m['u'], m['v'], m['p'], m['T'], m['tau']
            u_new, v_new, p_new, T_new, tau_new = actualizar_campos(
                presion_adversa, u, v, p, T,
                u.copy(), v.copy(), p.copy(), T.copy(),
                Re, Pr, Ec, Eu, dt_star, dx_star, dy_star, nx, ny
            )
            ensamble[e] = {'u': u_new, 'v': v_new, 'p': p_new, 'T': T_new, 'tau': tau_new}

        if observaciones_dict and n in observaciones_dict:
            t0 = time.time()
            obs_t = observaciones_dict[n]

            for var in variables_asimilar:
                x_phys = obs_t["x_phys"]
                x_idx = (x_phys * (nx - 1)).astype(int)
                y_idx = np.zeros_like(x_idx)
                Y_obs = obs_t[var]

                # Actualización punto por punto (más estable)
                for i, (xi, yi) in enumerate(zip(x_idx, y_idx)):
                    valores_ens = np.array([m[var][yi, xi] for m in ensamble])
                    var_ens = np.var(valores_ens)
                    if var_ens == 0:
                        continue  # evitar división por cero

                    # Ganancia de Kalman escalar
                    K = var_ens / (var_ens + sigma_obs**2)

                    for e in range(N_ens):
                        perturb = np.random.normal(0, sigma_obs)
                        innov = (Y_obs[i] + perturb) - ensamble[e][var][yi, xi]
                        ensamble[e][var][yi, xi] += K * innov

            total_assim_time += time.time() - t0

        if n in save_times:
            u_hist.append(np.mean([m['u'] for m in ensamble], axis=0))
            v_hist.append(np.mean([m['v'] for m in ensamble], axis=0))
            p_hist.append(np.mean([m['p'] for m in ensamble], axis=0))
            T_hist.append(np.mean([m['T'] for m in ensamble], axis=0))
            tau_hist.append(np.mean([m['tau'] for m in ensamble], axis=0))

    print("\n✅ Simulación EnKF finalizada.")

    return {
        "u_history": u_hist,
        "v_history": v_hist,
        "p_history": p_hist,
        "T_history": T_hist,
        "tau_history": tau_hist,
        "params": {
            "nx": nx, "ny": ny, "dt_star": dt_star, "nt": nt,
            "presion_adversa": presion_adversa,
            "N_ens": N_ens,
            "sigma_obs": sigma_obs,
            "variables_asimilar": variables_asimilar
        },
        "X_star": X_star,
        "Y_star": Y_star,
        "asimilacion_time": total_assim_time
    }


# 4D-Var

También necesita su propia funcion de simulacion. El proceso es totalmente diferente, al tener que buscar el estado que hace minima la funcion J

In [ ]:
def run_4dvar(nx=25, ny=25, presion_adversa=0.3,
                         observaciones_dict=None,
                         variables_asimilar=['T'],
                         N_FRAMES=300,
                         ventana_asimilacion=20,
                         max_iter=10,
                         alpha=0.05):
    """
    Simulación con asimilación de datos 4DVar (simplificada).

    Parámetros:
    -----------
    ventana_asimilacion : int
        Número de pasos dentro de cada ventana de optimización.
    max_iter : int
        Número de iteraciones de ajuste por ventana.
    alpha : float
        Paso de gradiente descendente.
    """

    # --- Parámetros físicos y temporales ---
    Re, Pr, Ec, Eu = 20.0, 10.0, 0.1, 1.0
    Lx_star = Ly_star = 1.0
    dx_star = Lx_star / (nx - 1)
    dy_star = Ly_star / (ny - 1)
    dt_cfl = 0.008 * dx_star / 1.0
    nt = int(np.ceil(2.0 / dt_cfl))
    dt_star = 1.0 / nt

    save_interval = max(1, nt // N_FRAMES)
    save_times = set(range(0, nt, save_interval))
    save_times.add(nt - 1)

    x_star = np.linspace(0, Lx_star, nx)
    y_star = np.linspace(0, Ly_star, ny)
    X_star, Y_star = np.meshgrid(x_star, y_star)

    # --- Condiciones iniciales ---
    def crear_estado_inicial():
        u = np.ones((ny, nx)) * 1.0
        v = np.zeros((ny, nx))
        p = np.zeros((ny, nx))
        T = np.zeros((ny, nx))
        tau = np.zeros((ny, nx))
        u[0, :] = -0.5
        u[-1, :] = 1.0
        return u, v, p, T, tau

    u_star, v_star, p_star, T_star, tau = crear_estado_inicial()

    u_hist, v_hist, p_hist, T_hist, tau_hist = [], [], [], [], []
    total_assim_time = 0.0

    n = 0
    while n < nt:
        t0 = time.time()

        # 1. Guardar estado inicial de la ventana
        u0, v0, p0, T0, _ = u_star.copy(), v_star.copy(), p_star.copy(), T_star.copy(), tau.copy()
        estados = []

        # 2. Simular hacia adelante
        for i in range(ventana_asimilacion):
            u_old, v_old, p_old, T_old = u_star.copy(), v_star.copy(), p_star.copy(), T_star.copy()
            u_star, v_star, p_star, T_star, tau = actualizar_campos(
                presion_adversa, u_old, v_old, p_old, T_old,
                u_star, v_star, p_star, T_star,
                Re, Pr, Ec, Eu, dt_star, dx_star, dy_star, nx, ny
            )
            estados.append((u_star.copy(), v_star.copy(), p_star.copy(), T_star.copy(), tau.copy()))

        # 3. Calcular gradiente e intentar mejorar T0 (simplificación: solo T)
        if observaciones_dict and metodo_observaciones_en_ventana(observaciones_dict, n, ventana_asimilacion):
            for _ in range(max_iter):
                T_adj = T0.copy()
                grad = np.zeros_like(T0)

                u_tmp, v_tmp, p_tmp, T_tmp = u0.copy(), v0.copy(), p0.copy(), T_adj.copy()
                for i in range(ventana_asimilacion):
                    u_tmp, v_tmp, p_tmp, T_tmp, _ = actualizar_campos(
                        presion_adversa, u_tmp.copy(), v_tmp.copy(), p_tmp.copy(), T_tmp.copy(),
                        u_tmp, v_tmp, p_tmp, T_tmp,
                        Re, Pr, Ec, Eu, dt_star, dx_star, dy_star, nx, ny
                    )
                    timestep = n + i
                    if timestep in observaciones_dict:
                        obs = observaciones_dict[timestep]
                        if 'T' in obs:
                            for x_phys, val in zip(obs['x_phys'], obs['T']):
                                j = int(x_phys * (nx - 1))
                                grad[0, j] += 2 * (T_tmp[0, j] - val)  # en y=0

                # Descenso de gradiente (solo en y=0)
                T0[0, :] -= alpha * grad[0, :]

            # Re-simular con T corregida
            u_star, v_star, p_star, T_star, tau = u0.copy(), v0.copy(), p0.copy(), T0.copy(), tau.copy()
            estados = []
            for i in range(ventana_asimilacion):
                u_old, v_old, p_old, T_old = u_star.copy(), v_star.copy(), p_star.copy(), T_star.copy()
                u_star, v_star, p_star, T_star, tau = actualizar_campos(
                    presion_adversa, u_old, v_old, p_old, T_old,
                    u_star, v_star, p_star, T_star,
                    Re, Pr, Ec, Eu, dt_star, dx_star, dy_star, nx, ny
                )
                estados.append((u_star.copy(), v_star.copy(), p_star.copy(), T_star.copy(), tau.copy()))

        total_assim_time += time.time() - t0

        # 4. Guardar
        for i in range(ventana_asimilacion):
            if (n + i) in save_times:
                u_hist.append(estados[i][0])
                v_hist.append(estados[i][1])
                p_hist.append(estados[i][2])
                T_hist.append(estados[i][3])
                tau_hist.append(estados[i][4])

        n += ventana_asimilacion

    print("\n✅ Simulación 4DVar finalizada.")

    return {
        "u_history": u_hist,
        "v_history": v_hist,
        "p_history": p_hist,
        "T_history": T_hist,
        "tau_history": tau_hist,
        "params": {
            "nx": nx, "ny": ny, "dt_star": dt_star, "nt": nt,
            "presion_adversa": presion_adversa,
            "ventana_asimilacion": ventana_asimilacion,
            "max_iter": max_iter
        },
        "X_star": X_star,
        "Y_star": Y_star,
        "asimilacion_time": total_assim_time
    }
def metodo_observaciones_en_ventana(observaciones_dict, n_ini, ventana):
    """Devuelve True si hay al menos una observación en la ventana."""
    return any((n_ini + i) in observaciones_dict for i in range(ventana))


# Generacion de datos

Se generan los 300 frames de la evolucion para una resolucion de 25x25 usando datos interpolados de la resolucion 800x800 como observaciones en la placa.

In [ ]:
EJECUTAR_SIMULACIONES = False  # 🔁 Cambia a True si quieres que esta celda se ejecute

if EJECUTAR_SIMULACIONES:

    # --- Cargar referencia ---
    with open("sim_normal/25_bloques.pkl", "rb") as f:
        ref_data = pickle.load(f)

    # --- Parámetros comunes ---
    resolucion = (25, 25)
    res_str = f"{resolucion[0]}"
    carpeta_base = f"DA/{res_str}_y_10%"
    os.makedirs(carpeta_base, exist_ok=True)

    metodos_especiales = ["enkf", "4dvar"]
    metodos_asimilacion = {
        "nudging": aplicar_nudging,
        "kf": aplicar_kalman_filter
    }

    variables_asimilar_set = [
    ["T"],         # solo temperatura
    ["p"],         # solo presión
    ["tau"],       # solo shear stress
    ["T", "p"],
    ["T", "tau"],
    ["p", "tau"],   # 🔧 Añadido: esta es la combinación "ps"
    ["T", "p", "tau"]
    ]

    n_sensores_set = [5, 10, 20]
    frecuencias_set = [300, 150, 70]

    # --- Bucle de configuraciones ---
    tag_map = {"T": "t", "p": "p", "tau": "s"}


    for variables_asimilar, n_sens, freq in product(variables_asimilar_set, n_sensores_set, frecuencias_set):

        print(f"\n🔧 Configuración: Vars={variables_asimilar}, Sensores={n_sens}, Freq={freq}")

        observaciones = extraer_observaciones(
        path_pkl="sim_normal/25_bloques.pkl",
        N_sens=n_sens,
        T_sens=freq,
        ruido_std=0.01,
        seed=42
    )

        observaciones_dict = {obs["t"]: obs for obs in observaciones}

        # ✅ Etiqueta segura para archivo
        vars_tag = ''.join(sorted(set(tag_map[v] for v in variables_asimilar)))

        # Métodos generales (nudging, kf)
        for nombre_metodo, funcion_asimilacion in metodos_asimilacion.items():
            print(f"🔄 Ejecutando {nombre_metodo.upper()}...")
            resultado = run_simulacion_general(
                nx=resolucion[0],
                ny=resolucion[1],
                presion_adversa=0.3,
                observaciones_dict=observaciones_dict,
                metodo_asimilacion=funcion_asimilacion,
                variables_asimilar=variables_asimilar,
                metodo_kwargs={},
                N_FRAMES=300
            )

            nombre_archivo = f"{nombre_metodo}_{res_str}_obs_{n_sens}_freq_{freq}_var_{vars_tag}.pkl"
            with open(os.path.join(carpeta_base, nombre_archivo), "wb") as f:
                pickle.dump(resultado, f)
            print(f"✅ Guardado: {nombre_archivo}")

        # EnKF
        print("🔄 Ejecutando ENKF...")
        resultado_enkf = run_enkf(
            nx=resolucion[0],
            ny=resolucion[1],
            N_ens=10,
            presion_adversa=0.3,
            observaciones_dict=observaciones_dict,
            variables_asimilar=variables_asimilar,
            sigma_obs=0.01,
        )
        nombre_enkf = f"enkf_{res_str}_obs_{n_sens}_freq_{freq}_var_{vars_tag}.pkl"
        with open(os.path.join(carpeta_base, nombre_enkf), "wb") as f:
            pickle.dump(resultado_enkf, f)
        print(f"✅ Guardado: {nombre_enkf}")

        # 4DVar
        print("🔄 Ejecutando 4DVAR...")
        resultado_4dvar = run_4dvar(
            nx=resolucion[0],
            ny=resolucion[1],
            presion_adversa=0.3,
            observaciones_dict=observaciones_dict,
            variables_asimilar=variables_asimilar,
            N_FRAMES=300,
            ventana_asimilacion=20,
            max_iter=10,
            alpha=0.05
        )
        nombre_4dvar = f"4dvar_{res_str}_obs_{n_sens}_freq_{freq}_var_{vars_tag}.pkl"
        with open(os.path.join(carpeta_base, nombre_4dvar), "wb") as f:
            pickle.dump(resultado_4dvar, f)
        print(f"✅ Guardado: {nombre_4dvar}")

# 23/07/2025 ---> tardó 388 minutos en ejecutar todo el script con las simulaciones con asimilación
# 13/08/2025 ---> tardó 491 minutos en ejecutar todo el script con las simulaciones con asimilación
# 14/08/2025 ---> tardó 383 minutos en ejecutar todo el script con las simulaciones con asimilación

